# Notebook 04 — PEGASUS Fine-tuning
Fine-tune google/pegasus-xsum on DialogSum, evaluate on test set.

In [ ]:
import sys, json
sys.path.insert(0, '..')

from src.data.loader import load_dialogsum
from src.methods.pegasus_finetune import train, predict, save_results
from src.evaluation.visualize import plot_training_curves


In [ ]:
dataset = load_dialogsum(cache_dir='../data/raw/dialogsum')
print(dataset)


In [ ]:
SKIP_TRAINING = False

if not SKIP_TRAINING:
    trainer = train(
        dataset,
        config_path='../configs/pegasus_config.yaml',
        output_dir='../models/pegasus_finetuned',
    )
    plot_training_curves(
        trainer.state.log_history,
        metric='rougeL',
        save_path='../results/figures/training_curves_pegasus.png',
        title='PEGASUS Fine-tuning — Loss & ROUGE-L',
    )


In [ ]:
test = dataset['test']
dialogues  = test['dialogue']
references = test['summary']

predictions = predict(
    dialogues,
    model_dir='../models/pegasus_finetuned',
    batch_size=8,
)
print('Sample prediction:', predictions[0])
print('Reference:        ', references[0])


In [ ]:
scores = save_results(predictions, references, '../results/scores/pegasus_results.json')
print('PEGASUS scores:', scores)
